# Hands-On Large Language Models: Language Understanding and Generation

In [2]:
%load_ext dotenv
%dotenv .env/hf.env


import torch
from torchinfo import summary
_model = "google/gemma-4-E4B-it"

# 1. An Introduction to Large Language Models/大语言模型导论

In [ ]:
# from transformers import AutoProcessor, AutoModelForCausalLM

# MODEL_ID = "google/gemma-4-E4B-it"

# # Load model
# processor = AutoProcessor.from_pretrained(MODEL_ID)
# model = AutoModelForCausalLM.from_pretrained(
#     MODEL_ID,
#     dtype="auto",
#     device_map="auto"
# )

# # Prompt
# messages = [
#     {"role": "system", "content": "You are a helpful assistant."},
#     {"role": "user", "content": "Write a short joke about saving RAM."},
# ]

# # Process input
# text = processor.apply_chat_template(
#     messages, 
#     tokenize=False, 
#     add_generation_prompt=True, 
#     enable_thinking=False
# )
# inputs = processor(text=text, return_tensors="pt").to(model.device)
# input_len = inputs["input_ids"].shape[-1]

# # Generate output
# outputs = model.generate(**inputs, max_new_tokens=1024)
# response = processor.decode(outputs[0][input_len:], skip_special_tokens=False)

# # Parse output
# processor.parse_response(response)

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.


{'role': 'assistant',
 'content': 'Why did the computer go to therapy?\n\nBecause it had too much **RAM** and needed to learn how to *de-clutter* its life! 💾😅'}

In [3]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    _model,
    # device_map="cuda",
    device_map="auto",
    torch_dtype="auto",
    # trust_remote_code=True,
)
tokenizer = AutoTokenizer.from_pretrained(_model)

# Create a pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False
)

# The prompt (user input / query)
messages = [
    {"role": "user", "content": "Create a funny joke about chickens."}
]
# Generate output
output = generator(messages)
print(output[0]["generated_text"])

Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The following generation flags are not valid and may be ignored: ['temperature', 'top_p', 'top_k']. Set `TRANSFORMERS_VERBOSITY=info` for more details.
[transformers] Both `max_new_tokens` (=500) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GemmaTokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress

Why did the chicken cross the playground?

To get to the *other slide*!<turn|>


# 2. Tokens and Embeddings/Token与嵌入

# 3. Looking Inside Large Language Models/探秘大语言模型内部

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

# 1. Load model and tokenizer
model = AutoModelForCausalLM.from_pretrained(
    _model,
    device_map = 'cpu',
    dtype=torch.bfloat16,
    low_cpu_mem_usage=True,
    local_files_only=True
)
tokenizer = AutoTokenizer.from_pretrained(_model)

[transformers] Detected torch version: 2.11.0
[transformers] Detected accelerate version: 1.13.0
[transformers] Detected torch version: 2.11.0
[transformers] loading configuration file config.json from cache at D:\software\huggingface\hub\models--google--gemma-4-E4B-it\snapshots\3555bddc93a623db8887dd2e52123facc45ade77\config.json
[transformers] text_config is None. Using default Gemma4TextConfig.
[transformers] vision_config is None. Gemma4Model.vision_tower will not be initialized.
[transformers] audio_config is None. Gemma4Model.audio_tower will not be initialized.
[transformers] Model config Gemma4Config {
  "architectures": [
    "Gemma4ForConditionalGeneration"
  ],
  "audio_config": {
    "_name_or_path": "",
    "architectures": null,
    "attention_chunk_size": 12,
    "attention_context_left": 13,
    "attention_context_right": 0,
    "attention_invalid_logits_value": -1000000000.0,
    "attention_logit_cap": 50.0,
    "chunk_size_feed_forward": 0,
    "conv_kernel_size": 5,


Loading weights:   0%|          | 0/2076 [00:00<?, ?it/s]

[transformers] loading configuration file generation_config.json from cache at D:\software\huggingface\hub\models--google--gemma-4-E4B-it\snapshots\3555bddc93a623db8887dd2e52123facc45ade77\generation_config.json
[transformers] Generate config GenerationConfig {
  "bos_token_id": 2,
  "do_sample": true,
  "eos_token_id": [
    1,
    106,
    50
  ],
  "pad_token_id": 0,
  "temperature": 1.0,
  "top_k": 64,
  "top_p": 0.95
}

[transformers] Offline mode: forcing local_files_only=True
[transformers] loading configuration file config.json from cache at D:\software\huggingface\hub\models--google--gemma-4-E4B-it\snapshots\3555bddc93a623db8887dd2e52123facc45ade77\config.json
[transformers] text_config is None. Using default Gemma4TextConfig.
[transformers] vision_config is None. Gemma4Model.vision_tower will not be initialized.
[transformers] audio_config is None. Gemma4Model.audio_tower will not be initialized.
[transformers] Model config Gemma4Config {
  "architectures": [
    "Gemma4ForCo

Gemma4ForConditionalGeneration(
  (model): Gemma4Model(
    (vision_tower): Gemma4VisionModel(
      (patch_embedder): Gemma4VisionPatchEmbedder(
        (input_proj): Linear(in_features=768, out_features=768, bias=False)
      )
      (encoder): Gemma4VisionEncoder(
        (rotary_emb): Gemma4VisionRotaryEmbedding()
        (layers): ModuleList(
          (0-15): 16 x Gemma4VisionEncoderLayer(
            (self_attn): Gemma4VisionAttention(
              (q_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (k_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (v_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=False)
              )
              (o_proj): Gemma4ClippableLinear(
                (linear): Linear(in_features=768, out_features=768, bias=Fals

NameError: name 'model' is not defined

In [ ]:
# Create a pipeline
generator = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer,
    return_full_text=False,
    max_new_tokens=500,
    do_sample=False
)

model = torch.compile(model, mode="reduce-overhead")

In [ ]:


prompt = "Write an email apologizing to Sarah for the tragic gardening mishap. Explain how it happened."
output = generator(prompt)
print(output[0]['generated_text'])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens', 'do_sample'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] The model 'OptimizedModule' is not supported for text-generation. Supported models are ['PeftModelForCausalLM', 'AfmoeForCausalLM', 'ApertusForCausalLM', 'ArceeForCausalLM', 'AriaTextForCausalLM', 'BambaForCausalLM', 'BartForCausalLM', 'BertLMHeadModel', 'BertGenerationDecoder', 'BigBirdForCausalLM', 'BigBirdPegasusForCausalLM', 'BioGptForCausalLM', 'BitNetForCausalLM', 'BlenderbotForCausalLM', 'BlenderbotSmallForCausalLM', 'BloomForCausalLM', 'BltForCausalLM', 'CamembertForCausalLM', 'CodeGenForCausalLM', 'CohereForCausalLM', 'Cohere2ForCausalLM', 'CpmAntForCausalLM', 'CTRLLMHeadModel', 'CwmForCausalLM', 'Data2VecTextForCausalLM', 'DbrxForCausalLM', 'DeepseekV2ForCausalLM', 'DeepseekV3ForCausalLM',

 Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was. Explain how it was.

# 4. Text Classification/文本分类

# 5. Text Clustering and Topic Modeling/文本聚类与主题建模

# 6. Prompt Engineering/提示词工程

# 7. Advanced Text Generation Techniques and Tools/高级文本生成技术与工具

# 8. Semantic Search and Retrieval-Augmented Generation/语义搜索与检索增强生成

# 9. Multimodal Large Language Models/多模态大语言模型

# 10. Creating Text Embedding Models/构建文本嵌入模型

# 11. Fine-Tuning Representation Models for Classification/微调表示模型以用于分类任务

# 12. Fine-Tuning Generation Models/微调生成模型

# Cleanup